# Video Food Classifier -- ViT-Mobile XXS + BiLSTM

Fine-grained video/image classification model for food content moderation.
Classifies into **16 categories** across 3 health groups, using a MobileViT-XXS
backbone with bidirectional LSTM temporal pooling.

**Classes (16):**
- **Healthy (8):** fruits, vegetables, salads, seafood, grilled\_meat, grain\_bowls, soups, smoothies
- **Unhealthy (7):** burgers, pizza, fried\_food, desserts, candy\_sweets, salty\_snacks, sugary\_drinks
- **Not food (1):** not\_food

**Dataset:** ~8,600 images across 16 classes (~540 per class)

Steps covered:
1. Setup (clone repo, install deps)
2. Environment check
3. Data download (16 classes, ~8,600 images)
4. Dataset inspection
5. Training (MobileViT-XXS + BiLSTM, 20 epochs)
6. Evaluation on test split
7. Model validation (leakage audit, k-fold CV)
8. Inference demo
9. Health-label rollup (fine-grained -> healthy / unhealthy / not\_food)
10. Export to mobile formats
11. Upload to Hugging Face Hub

## 1. Setup -- Clone Repo & Install Dependencies

In [ ]:
import os

# Clone the repository (skip if already cloned)
REPO_URL = "https://github.com/whispr-messenger/moderation-service.git"
REPO_DIR = "/content/moderation-service"
BRANCH = "WHISPR-668"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already cloned at {REPO_DIR}")

# Set working directory to the vit_video package
VIT_DIR = os.path.join(REPO_DIR, "src", "vit_video")
os.chdir(VIT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q icrawler

## 2. Environment Check

In [ ]:
import os, sys, importlib
from pathlib import Path

# Re-establish paths (survives kernel restart)
REPO_DIR = "/content/moderation-service"
VIT_DIR = os.path.join(REPO_DIR, "src", "vit_video")
os.chdir(VIT_DIR)

print(f"Python {sys.version}")

for pkg in ["torch", "torchvision", "timm", "cv2", "sklearn", "icrawler"]:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, "__version__", "ok")
        print(f"  {pkg:16s} {ver}")
    except ImportError:
        print(f"  {pkg:16s} NOT INSTALLED")

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Add src/ to sys.path so vit_video package imports work
import sys
from pathlib import Path

SRC_DIR = str(Path(VIT_DIR).parent)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from vit_video.utils.hardware import get_device, print_device_info

device = get_device()
print_device_info(hint_cuda=True)

## 3. Data Download & Frame Extraction

Downloads images from Bing for **16 fine-grained classes** (8 healthy, 7 unhealthy,
1 not-food). Each search keyword becomes a pseudo-video. ~27 keywords per class,
20 images each = **~8,600 images total**.

In [ ]:
from vit_video.paths import DEFAULT_DATASET_DIR, DEFAULT_FRAMES_DIR
from vit_video.data.splits import frames_directory_has_images

DATASET_DIR = DEFAULT_DATASET_DIR
FRAMES_DIR  = DEFAULT_FRAMES_DIR

# ---------------------------------------------------------------------------
# Health-label mapping: fine-grained class -> moderation group
# ---------------------------------------------------------------------------
HEALTH_LABELS = {
    # Healthy
    "fruits":        "healthy",
    "vegetables":    "healthy",
    "salads":        "healthy",
    "seafood":       "healthy",
    "grilled_meat":  "healthy",
    "grain_bowls":   "healthy",
    "soups":         "healthy",
    "smoothies":     "healthy",
    # Unhealthy
    "burgers":       "unhealthy",
    "pizza":         "unhealthy",
    "fried_food":    "unhealthy",
    "desserts":      "unhealthy",
    "candy_sweets":  "unhealthy",
    "salty_snacks":  "unhealthy",
    "sugary_drinks": "unhealthy",
    # Not food
    "not_food":      "not_food",
}

# ---------------------------------------------------------------------------
# 16-class dataset: ~27 keywords per class x 20 images = ~540 images/class
# ---------------------------------------------------------------------------
FOOD_CATEGORIES = {
    # ======================== HEALTHY (8 classes) ========================
    "fruits": [
        "banana fruit close up", "fresh apple fruit plate",
        "fresh orange citrus fruit", "fresh mixed berries bowl",
        "mango sliced fruit tropical", "watermelon slice summer",
        "kiwi fruit halved green", "pineapple sliced tropical",
        "grapes bunch fresh fruit", "pomegranate seeds bowl",
        "papaya fruit sliced", "blueberries bowl fresh",
        "strawberries fresh basket", "peach fruit sliced",
        "pear fruit fresh green", "cherry fruit bowl",
        "plum fruit purple fresh", "cantaloupe melon sliced",
        "passion fruit halved", "dragon fruit sliced pink",
        "fig fruit fresh cut", "apricot fruit orange",
        "raspberry bowl fresh red", "tangerine citrus peeled",
        "coconut halved fresh white", "grapefruit halved citrus",
        "avocado halved fresh green",
    ],
    "vegetables": [
        "broccoli vegetable steamed", "steamed vegetables plate",
        "roasted vegetables oven tray", "vegetable stir fry wok",
        "grilled zucchini vegetables", "sauteed mushrooms pan",
        "asparagus grilled plate", "bell pepper sliced colorful",
        "cauliflower roasted golden", "brussels sprouts roasted",
        "spinach sauteed garlic", "green beans steamed plate",
        "sweet corn cob grilled", "eggplant grilled sliced",
        "artichoke cooked plate", "beets roasted sliced",
        "carrots roasted honey", "cabbage sauteed dish",
        "kale cooked sauteed", "peas green bowl fresh",
        "radish sliced fresh salad", "turnip roasted root",
        "celery sticks fresh snack", "bok choy stir fry",
        "snap peas fresh green", "okra cooked vegetable",
        "leek sauteed sliced",
    ],
    "salads": [
        "green salad bowl meal", "kale spinach salad bowl",
        "cucumber tomato salad fresh", "caesar salad croutons parmesan",
        "mixed greens salad dressing", "greek salad feta olives",
        "caprese salad mozzarella tomato", "quinoa salad vegetables",
        "asian sesame salad bowl", "waldorf salad apple walnut",
        "cobb salad eggs bacon avocado", "arugula salad shaved parmesan",
        "tabbouleh parsley bulgur salad", "pasta salad vegetables cold",
        "coleslaw cabbage salad", "beet goat cheese salad",
        "chickpea salad mediterranean", "nicoise salad tuna",
        "fruit salad mixed colorful", "potato salad picnic",
        "southwest salad corn beans", "thai mango salad spicy",
        "seaweed salad japanese", "fattoush salad middle eastern",
        "spinach strawberry salad", "broccoli salad cranberry",
        "lentil salad warm herbs",
    ],
    "seafood": [
        "grilled salmon vegetables plate", "sushi roll fresh fish",
        "baked cod fish lemon", "grilled shrimp skewers",
        "poke bowl fresh tuna", "sashimi platter japanese",
        "fish tacos fresh cilantro", "steamed mussels garlic wine",
        "lobster tail grilled butter", "scallops seared pan",
        "grilled tuna steak sesame", "shrimp cocktail appetizer",
        "crab legs steamed plate", "oysters fresh half shell",
        "calamari grilled not fried", "tilapia baked herbs lemon",
        "ceviche fresh seafood lime", "smoked salmon bagel cream",
        "sardines grilled Mediterranean", "sea bass baked whole",
        "trout pan seared butter", "clam chowder soup bowl",
        "fish curry coconut bowl", "prawn stir fry vegetables",
        "anchovy pizza topping", "mackerel grilled Japanese",
        "octopus grilled tentacles",
    ],
    "grilled_meat": [
        "grilled chicken breast vegetables", "turkey breast sliced lean",
        "hard boiled eggs protein", "grilled steak medium rare",
        "lamb chops grilled herbs", "pork tenderloin roasted",
        "chicken kebab skewers grilled", "beef stir fry vegetables",
        "rotisserie chicken whole", "meatballs turkey lean",
        "chicken thighs grilled crispy", "sirloin steak grilled plate",
        "venison steak cooked rare", "duck breast seared skin",
        "beef brisket smoked sliced", "chicken wings baked not fried",
        "pork chops grilled bone", "flank steak marinated sliced",
        "rabbit roasted herbs", "quail grilled whole small",
        "beef jerky dried strips", "chicken satay peanut sauce",
        "shawarma chicken wrap plate", "pulled pork sandwich",
        "filet mignon steak dinner", "rack of lamb roasted",
        "beef tartare raw prepared",
    ],
    "grain_bowls": [
        "quinoa bowl healthy meal", "brown rice bowl vegetables",
        "chickpea buddha bowl", "overnight oats fruit jar",
        "oatmeal breakfast bowl berries", "whole grain toast avocado",
        "granola yogurt bowl healthy", "couscous vegetable bowl",
        "farro grain bowl warm", "barley soup healthy grain",
        "acai bowl fruit toppings", "chia pudding healthy dessert",
        "rice cakes healthy snack", "cottage cheese fruit bowl",
        "tofu vegetable rice bowl", "black bean rice bowl",
        "poke rice bowl fresh", "burrito bowl healthy rice",
        "bibimbap korean rice bowl", "risotto mushroom arborio",
        "congee rice porridge asian", "polenta creamy bowl",
        "muesli breakfast bowl milk", "bulgur wheat bowl",
        "wild rice pilaf herbs", "tabbouleh grain bowl",
        "hummus grain bowl plate",
    ],
    "soups": [
        "lentil soup healthy bowl", "miso soup japanese tofu",
        "vegetable soup homemade pot", "minestrone soup italian",
        "tomato soup grilled cheese", "chicken noodle soup bowl",
        "butternut squash soup creamy", "gazpacho cold soup spanish",
        "pho vietnamese noodle soup", "wonton soup dumplings broth",
        "corn chowder soup bowl", "mushroom soup creamy bowl",
        "black bean soup mexican", "split pea soup ham",
        "french onion soup cheese", "egg drop soup chinese",
        "coconut curry soup thai", "borscht beet soup red",
        "hot and sour soup chinese", "ramen noodle soup japanese",
        "tortilla soup mexican", "broccoli cheddar soup bowl",
        "lobster bisque creamy", "pozole mexican pork soup",
        "laksa noodle soup spicy", "udon noodle soup japanese",
        "bone broth clear soup",
    ],
    "smoothies": [
        "green smoothie drink healthy", "berry smoothie purple blend",
        "mango smoothie tropical drink", "banana smoothie creamy glass",
        "protein smoothie post workout", "acai smoothie bowl thick",
        "spinach kale green smoothie", "strawberry banana smoothie",
        "peanut butter smoothie protein", "avocado smoothie green creamy",
        "coconut smoothie tropical blend", "blueberry smoothie antioxidant",
        "oat milk smoothie breakfast", "detox green juice cleanse",
        "carrot orange juice fresh", "beet juice red healthy",
        "watermelon juice fresh summer", "celery juice green health",
        "ginger turmeric shot wellness", "matcha latte green tea",
        "kombucha fermented tea drink", "fresh pressed orange juice",
        "cucumber mint water infused", "lemon water detox morning",
        "iced green tea unsweetened", "coconut water fresh natural",
        "almond milk smoothie vanilla",
    ],

    # ======================== UNHEALTHY (7 classes) ========================
    "burgers": [
        "cheeseburger close up eating", "bacon cheeseburger fast food",
        "double cheeseburger stacked", "big mac hamburger meal",
        "slider burgers mini cheese", "mushroom swiss burger",
        "bbq bacon burger loaded", "wagyu beef burger gourmet",
        "turkey burger fast food", "veggie burger fast food chain",
        "patty melt burger griddled", "western burger onion ring",
        "jalapeÃ±o burger spicy cheese", "blue cheese burger gourmet",
        "smash burger crispy thin", "whopper burger king meal",
        "five guys burger fries", "in n out burger animal style",
        "shake shack burger meal", "breakfast burger egg bacon",
        "truffle burger gourmet fancy", "chipotle burger spicy sauce",
        "teriyaki burger pineapple", "elk burger wild game",
        "bison burger gourmet meat", "lamb burger mediterranean",
        "hot dog mustard ketchup bun",
    ],
    "pizza": [
        "pepperoni pizza slice cheese", "margherita pizza fresh basil",
        "meat lovers pizza loaded", "hawaiian pizza ham pineapple",
        "deep dish pizza chicago style", "thin crust pizza new york",
        "stuffed crust pizza cheese", "buffalo chicken pizza",
        "bbq chicken pizza slice", "white pizza garlic ricotta",
        "supreme pizza everything toppings", "four cheese pizza blend",
        "mushroom truffle pizza gourmet", "veggie pizza loaded toppings",
        "calzone folded pizza pocket", "pizza rolls frozen snack",
        "flatbread pizza crispy thin", "sicilian pizza thick square",
        "detroit style pizza crispy", "neapolitan pizza wood fired",
        "french bread pizza baked", "bagel bites pizza snack",
        "pizza hut delivery box", "dominos pizza delivery",
        "frozen pizza oven baked", "garlic knots pizza side",
        "pizza dough stretching making",
    ],
    "fried_food": [
        "french fries fast food", "fried chicken bucket crispy",
        "deep fried chicken wings", "fried onion rings basket",
        "mozzarella sticks fried cheese", "deep fried corn dog",
        "fried fish and chips", "chicken nuggets dipping sauce",
        "fried calamari appetizer", "tempura fried battered shrimp",
        "jalapeno poppers fried cheese", "deep fried oreos fair food",
        "fried pickles appetizer", "fried cheese curds basket",
        "chicken tenders fried crispy", "fried wontons crispy golden",
        "hush puppies fried cornmeal", "fried zucchini sticks breaded",
        "corn fritters fried golden", "fried shrimp basket tartar",
        "tater tots crispy fried", "fried ravioli appetizer",
        "poutine fries gravy cheese", "loaded fries cheese bacon",
        "curly fries seasoned crispy", "waffle fries chick fil a",
        "sweet potato fries fried",
    ],
    "desserts": [
        "chocolate cake slice frosting", "glazed donut dessert sprinkles",
        "cupcake frosting sprinkles", "cheesecake slice strawberry",
        "churros fried sugar cinnamon", "cinnamon roll frosting icing",
        "brownie chocolate fudge walnut", "pancakes syrup butter stack",
        "funnel cake powdered sugar", "cream puff pastry custard",
        "tiramisu dessert coffee cream", "pie slice whipped cream",
        "waffle chocolate syrup cream", "eclair chocolate pastry cream",
        "croissant butter flaky pastry", "baklava honey pistachio pastry",
        "apple pie lattice crust", "red velvet cake cream cheese",
        "birthday cake frosting candles", "creme brulee torch caramel",
        "panna cotta dessert italian", "profiteroles chocolate cream",
        "danish pastry fruit icing", "scone clotted cream jam",
        "tres leches cake soaked", "bread pudding caramel sauce",
        "glazed donuts box dozen",
    ],
    "candy_sweets": [
        "candy sweets unwrapping colorful", "chocolate bar wrapper bite",
        "gummy bears candy bowl", "jelly beans colorful candy",
        "cotton candy carnival pink", "caramel popcorn sweet snack",
        "lollipop candy colorful stick", "ice cream sundae chocolate",
        "ice cream cone scoops", "frozen yogurt toppings cup",
        "popsicle frozen fruit bar", "milkshake thick straw glass",
        "chocolate truffles assorted box", "fudge chocolate squares plate",
        "marshmallow smores campfire", "toffee candy wrapped butter",
        "taffy saltwater candy pulled", "rock candy crystal sugar",
        "candy corn halloween bowl", "licorice red black candy",
        "chocolate mousse dessert rich", "M&Ms candy colorful bowl",
        "skittles candy rainbow bag", "kit kat chocolate wafer",
        "snickers bar chocolate peanut", "reeses peanut butter cup",
        "gummy worms sour candy",
    ],
    "salty_snacks": [
        "potato chips bag eating", "nachos cheese jalapeno loaded",
        "cheese puffs snack orange", "pretzel bites cheese dip",
        "corn chips salsa queso", "popcorn butter movie snack",
        "trail mix salty sweet nuts", "crackers cheese plate snack",
        "doritos chips nacho cheese", "cheetos flaming hot bag",
        "pringles chips can stack", "tortilla chips guacamole",
        "goldfish crackers snack bowl", "chex mix snack bag party",
        "beef jerky dried snack", "pork rinds fried snack",
        "sunflower seeds snack salted", "mixed nuts salted roasted",
        "rice crackers asian snack", "cheese crackers snack pack",
        "onion dip chips snack", "hummus chips snack plate",
        "breadsticks garlic butter", "cheese balls fried orange",
        "bugles cone shaped snack", "funyuns onion ring snack",
        "combos pretzel cheese snack",
    ],
    "sugary_drinks": [
        "soda cola pouring glass", "milkshake dessert drink straw",
        "energy drink can neon", "iced frappuccino whipped cream",
        "bubble tea boba pearls", "slurpee frozen drink colorful",
        "lemonade sweet pitcher summer", "sweet iced tea glass",
        "hot chocolate whipped cream", "mocha coffee sweet cream",
        "fruit punch bowl red drink", "capri sun juice pouch",
        "kool aid pitcher colorful", "mountain dew soda green",
        "sprite soda lemon lime", "fanta orange soda can",
        "root beer float ice cream", "cream soda vanilla drink",
        "dr pepper soda can glass", "gatorade sports drink bottle",
        "arizona iced tea can", "starbucks caramel frappuccino",
        "chocolate milk glass straw", "strawberry milkshake pink",
        "vanilla milkshake thick glass", "mango lassi sweet drink",
        "pina colada cocktail sweet",
    ],

    # ======================== NOT FOOD (1 class) ========================
    "not_food": [
        # Animals
        "cat sleeping couch home", "dog playing park grass",
        "bird flying blue sky", "horse running field pasture",
        "fish aquarium swimming", "rabbit pet fluffy cute",
        # People & activities
        "person walking street city", "people jogging park morning",
        "crowd concert music event", "children playground school",
        "office workers meeting room", "students classroom lecture",
        # Vehicles
        "car driving highway road", "bicycle riding path urban",
        "airplane flying sky clouds", "train station platform",
        # Nature
        "sunset ocean beach landscape", "mountain hiking trail nature",
        "flowers garden spring bloom", "snow winter trees forest",
        "waterfall tropical forest", "desert sand dunes landscape",
        # Urban
        "city skyline night lights", "construction site crane building",
        "shopping mall interior stores", "bridge architecture river",
        # Objects
        "book reading library study", "laptop computer desk work",
        "guitar music instrument", "painting art museum gallery",
        "phone texting screen hands", "camera equipment lens photo",
        # Sports
        "soccer ball field sports", "basketball court game playing",
        "swimming pool water blue", "yoga mat exercise fitness",
        # Food-adjacent (hard negatives)
        "empty plate table setting", "kitchen counter clean modern",
        "cutting board knife wooden", "cooking pot stove kitchen",
        "grocery store aisle shelves", "restaurant interior empty tables",
        "menu card restaurant dining", "vending machine snacks exterior",
        "kitchen utensils drawer", "oven stove kitchen appliance",
        "refrigerator open empty", "coffee machine espresso maker",
        "wine glasses empty table", "barbeque grill empty outdoor",
        "dining table set no food",
    ],
}

print(f"{'Class':<16s} {'Keywords':>8s}  {'~Images':>8s}  Health")
print("-" * 50)
for cat, kws in FOOD_CATEGORIES.items():
    print(f"  {cat:<14s} {len(kws):>6d}    {len(kws)*20:>6d}   {HEALTH_LABELS[cat]}")
print(f"\n  {'TOTAL':<14s} {sum(len(v) for v in FOOD_CATEGORIES.values()):>6d}    "
      f"{sum(len(v) for v in FOOD_CATEGORIES.values())*20:>6d}")

In [ ]:
import re, shutil, logging
from icrawler.builtin import BingImageCrawler

IMAGES_PER_KEYWORD = 20

# Classes where we do NOT append "food" to the search query
NON_FOOD_CLASSES = {"not_food"}


def download_images(frames_dir, categories, images_per_keyword=20):
    for category, keywords in categories.items():
        cat_dir = frames_dir / category
        cat_dir.mkdir(parents=True, exist_ok=True)

        for keyword in keywords:
            slug = re.sub(r"[^a-z0-9]+", "_", keyword.lower()).strip("_")
            existing = list(cat_dir.glob(f"{slug}_frame_*.jpg"))
            if len(existing) >= images_per_keyword:
                continue

            tmp_dir = frames_dir / "_tmp_download"
            if tmp_dir.exists():
                shutil.rmtree(tmp_dir)
            tmp_dir.mkdir()

            try:
                query = keyword if category in NON_FOOD_CLASSES else keyword + " food"
                crawler = BingImageCrawler(
                    storage={"root_dir": str(tmp_dir)},
                    log_level=logging.WARNING,
                )
                crawler.crawl(keyword=query, max_num=images_per_keyword)

                idx = 0
                for img_file in sorted(tmp_dir.iterdir()):
                    if img_file.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"):
                        dest = cat_dir / f"{slug}_frame_{idx:04d}.jpg"
                        shutil.move(str(img_file), str(dest))
                        idx += 1
            except Exception as e:
                print(f"  [WARN] {category}/{keyword}: {e}")
            finally:
                if tmp_dir.exists():
                    shutil.rmtree(tmp_dir)

        n_frames = len(list(cat_dir.glob("*.jpg")))
        print(f"  {category}: {n_frames} images")


if not frames_directory_has_images(FRAMES_DIR):
    print("Downloading images (this takes ~20-30 min on Colab)...")
    download_images(FRAMES_DIR, FOOD_CATEGORIES, IMAGES_PER_KEYWORD)
    print("\nDownload complete.")
else:
    print(f"Frames already exist in {FRAMES_DIR} -- skipping download.")

## 4. Dataset Inspection

In [ ]:
from vit_video.data.splits import discover_videos_by_class, count_frame_images

videos_by_class = discover_videos_by_class(FRAMES_DIR)
total_frames = count_frame_images(FRAMES_DIR)

print(f"Total frames on disk: {total_frames}\n")
for cls, stems in sorted(videos_by_class.items()):
    print(f"  {cls}: {len(stems)} unique videos")

In [ ]:
# Visualise sample frames
%matplotlib inline
import matplotlib.pyplot as plt
from vit_video.data.dataset import VideoDataset

preview_ds = VideoDataset(
    root=FRAMES_DIR,
    frames_per_video=8,
    img_size=224,
    augment=False,
)
print(f"Dataset size: {len(preview_ds)} videos, classes: {preview_ds.classes}")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
video_tensor, label = preview_ds[0]
for i, ax in enumerate(axes.flat):
    frame = video_tensor[i].permute(1, 2, 0).numpy()
    frame = frame * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]  # un-normalise
    frame = frame.clip(0, 1)
    ax.imshow(frame)
    ax.set_title(f"Frame {i}")
    ax.axis("off")
fig.suptitle(f"Sample video -- class: {preview_ds.classes[label]}", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Training

Uses `train.py` via its Python API.
- **Backbone:** ViT-B/16 on GPU, MobileViT-XXS on CPU
- **Temporal pooling:** Bidirectional LSTM (1-layer, hidden=feat\_dim/2)
- **20 epochs**, batch 8, lr 3e-5, dropout 0.4
- AdamW + warmup (3 epochs) + cosine annealing
- Label smoothing 0.1, gradient clipping 1.0
- Class weighting enabled (handles imbalance across 16 classes)

In [ ]:
from vit_video.data.splits import ensure_split_manifest

# Create or reuse a video-level split manifest
manifest_path = ensure_split_manifest(
    frames_root=FRAMES_DIR,
    manifest_path=DATASET_DIR / "video_split_manifest.json",
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
)
print(f"Split manifest: {manifest_path}")

In [ ]:
import argparse
from vit_video.train import main as train_main
from vit_video.paths import PACKAGE_ROOT

MODEL_PATH = PACKAGE_ROOT / "models" / "best_food_classifier.pth"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

train_args = argparse.Namespace(
    dataset_dir=str(FRAMES_DIR),
    epochs=20,
    batch_size=8,
    lr=3e-5,
    weight_decay=1e-3,
    max_grad_norm=1.0,
    dropout=0.4,
    num_frames=8,
    img_size=224,
    disable_augmentation=False,
    class_weighting=True,
    min_samples_per_class=20,
    temporal_pool="lstm",
    norm_mean="0.485,0.456,0.406",
    norm_std="0.229,0.224,0.225",
    hparam_search_epochs=0,
    lr_candidates="5e-6,1e-5,3e-5,5e-5,1e-4",
    num_workers=2,
    patience=7,
    min_delta=5e-5,
    backbone="auto",
    output_model=str(MODEL_PATH),
    resume_from="",
    split_manifest=str(manifest_path),
    no_auto_split_manifest=True,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    split_seed=42,
)

train_main(train_args)

model_path = str(MODEL_PATH)
print(f"\nSaved model: {model_path}")

### Plot training history

In [ ]:
import json
import matplotlib.pyplot as plt

history_file = Path(model_path).with_name(
    Path(model_path).stem + "_history.json"
)
if history_file.exists():
    history = json.loads(history_file.read_text())
else:
    print(f"History file not found: {history_file}")
    history = None

if history:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history["train_loss"], label="Train", marker="o", markersize=4)
    ax1.plot(history["val_loss"], label="Val", marker="s", markersize=4)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history["train_acc"], label="Train", marker="o", markersize=4)
    ax2.plot(history["val_acc"], label="Val", marker="s", markersize=4)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title("Accuracy")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 6. Evaluation on Test Split

In [ ]:
from vit_video.test import build_test_loader, evaluate, print_results, save_results
from vit_video.utils.model_utils import load_model_from_checkpoint

test_loader, classes, _test_indices = build_test_loader(
    dataset_root=str(FRAMES_DIR),
    batch_size=4,
    num_frames=8,
    num_workers=2,
    split_manifest=str(manifest_path),
)

NUM_CLASSES = len(classes)
model = load_model_from_checkpoint(model_path, num_classes=NUM_CLASSES, device=device)

results = evaluate(model, test_loader, device, classes)
print_results(results)
save_results(results, PACKAGE_ROOT / "results")

In [ ]:
# Confusion matrix (sized for 16 classes)
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cm = np.array(results["confusion_matrix"])
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix ({NUM_CLASSES} classes)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"\nAccuracy:  {results['accuracy']:.4f}")
print(f"Precision: {results['precision_macro']:.4f}")
print(f"Recall:    {results['recall_macro']:.4f}")
print(f"F1:        {results['f1_macro']:.4f}")

## 7. Model Validation

Runs data-leakage audit, k-fold cross-validation, and external-data testing.

In [ ]:
# Run the full validation suite via CLI
import subprocess, sys

validate_script = str(PACKAGE_ROOT / "validate_model.py")

cmd = [
    sys.executable, validate_script,
    "--model", str(model_path),
    "--dataset-dir", str(FRAMES_DIR),
    "--n-folds", "3",        # fewer folds for speed in notebook
    "--epochs", "5",         # shorter CV epochs
    "--skip-external",       # remove this flag to also test on fresh YouTube data
]
subprocess.run(cmd, check=False)

## 8. Inference Demo

In [ ]:
from PIL import Image
from vit_video.utils.data_utils import build_transform
import time

transform = build_transform(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

# Test on one image from each health group
test_dirs = ["fruits", "burgers", "not_food"]
model.eval()

fig, axes = plt.subplots(1, len(test_dirs), figsize=(5 * len(test_dirs), 5))
for ax, cls_name in zip(axes, test_dirs):
    sample = sorted(FRAMES_DIR.glob(f"{cls_name}/*.jpg"))[:1]
    if not sample:
        ax.set_title(f"{cls_name}: no images")
        ax.axis("off")
        continue

    img = Image.open(sample[0]).convert("RGB").resize((224, 224))
    video_tensor = torch.stack([transform(img)] * 8).unsqueeze(0).to(device)

    with torch.no_grad():
        start = time.perf_counter()
        outputs = model(video_tensor)
        ms = (time.perf_counter() - start) * 1000.0
        probs = torch.softmax(outputs, dim=1)
        idx = torch.argmax(probs, dim=1).item()
        conf = probs[0][idx].item()

    pred_class = classes[idx]
    health = HEALTH_LABELS.get(pred_class, "?")
    ax.imshow(img)
    ax.set_title(f"{pred_class} ({conf:.0%})\n[{health}] {ms:.0f}ms")
    ax.axis("off")
plt.suptitle("Inference Demo", fontsize=14)
plt.tight_layout()
plt.show()

## 9. Health-Label Rollup

Maps the 16 fine-grained predictions to 3 moderation groups:
**healthy**, **unhealthy**, **not\_food**.

In [ ]:
# Roll up test predictions to health groups
from collections import Counter

preds  = results["predictions"]
truths = results["ground_truth"]

health_preds  = [HEALTH_LABELS.get(classes[p], "?") for p in preds]
health_truths = [HEALTH_LABELS.get(classes[t], "?") for t in truths]

health_correct = sum(p == t for p, t in zip(health_preds, health_truths))
health_acc = health_correct / len(health_preds) if health_preds else 0

print(f"Health-level accuracy: {health_acc:.2%}  ({health_correct}/{len(health_preds)})\n")

# Confusion at health level
from sklearn.metrics import confusion_matrix as sk_cm, classification_report
health_labels = ["healthy", "not_food", "unhealthy"]
h_cm = sk_cm(health_truths, health_preds, labels=health_labels)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(h_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=health_labels, yticklabels=health_labels, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Health-Level Confusion Matrix (acc={health_acc:.1%})")
plt.tight_layout()
plt.show()

print(classification_report(health_truths, health_preds, labels=health_labels, zero_division=0))

## 10. Export to Mobile Formats

In [ ]:
import subprocess, sys

export_script = str(PACKAGE_ROOT / "export_mobile.py")
export_dir = PACKAGE_ROOT / "exported_models"
results_file = PACKAGE_ROOT / "results" / "test_results.json"

export_cmd = [
    sys.executable, export_script,
    "--model", str(model_path),
    "--output-dir", str(export_dir),
    "--format", "torchscript", "onnx",
    "--num-classes", str(NUM_CLASSES),
    "--classes", ",".join(classes),
    "--eval-results", str(results_file),
]
subprocess.run(export_cmd, check=False)

print("\nExported files:")
for f in sorted(export_dir.glob("*")):
    size_mb = f.stat().st_size / 1024 / 1024 if f.is_file() else 0
    print(f"  {f.name:40s} {size_mb:>7.1f} MB" if f.is_file() else f"  {f.name}/")

## 11. Upload to Hugging Face Hub

Uploads dataset and exported model to HuggingFace.

In [ ]:
# Load HF token from .env
import os
from pathlib import Path

env_path = Path(REPO_DIR) / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()

hf_token = os.environ.get("HF_TOKEN", "")
if hf_token and hf_token != "your_new_token_here":
    !huggingface-cli login --token {hf_token}
else:
    print("No HF_TOKEN in .env -- run: huggingface-cli login")

In [ ]:
# Upload dataset to Hugging Face
from huggingface_hub import HfApi, create_repo

HF_DATASET_REPO = "maia2000/food-classifier-dataset"
create_repo(HF_DATASET_REPO, repo_type="dataset", exist_ok=True)

api = HfApi()
api.upload_folder(
    repo_id=HF_DATASET_REPO, repo_type="dataset",
    folder_path=str(FRAMES_DIR), path_in_repo="frames",
    commit_message="Upload 16-class food classification dataset",
)
manifest_file = DATASET_DIR / "video_split_manifest.json"
if manifest_file.exists():
    api.upload_file(
        repo_id=HF_DATASET_REPO, repo_type="dataset",
        path_or_fileobj=str(manifest_file),
        path_in_repo="video_split_manifest.json",
        commit_message="Upload split manifest",
    )
print(f"Dataset: https://huggingface.co/datasets/{HF_DATASET_REPO}")


In [ ]:
# Upload model to Hugging Face
import subprocess, sys

upload_script = str(PACKAGE_ROOT / "upload_hf.py")
subprocess.run([
    sys.executable, upload_script,
    "--repo-id", "maia2000/food-classifier",
    "--export-dir", str(export_dir),
], check=False)


## Model Summary

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model Summary")
print(f"  Architecture:       MobileViTModel + BiLSTM")
print(f"  Backbone:           {getattr(model, "model_name", "auto")}")
print(f"  Temporal pooling:   {getattr(model, "temporal_pool", "lstm")}")
print(f"  Total parameters:   {total_params:,}")
print(f"  Trainable params:   {trainable:,}")
print(f"  Input shape:        (B, 8, 3, 224, 224)")
print(f"  Output:             (B, {NUM_CLASSES}) {classes}")
print(f"  Device:             {device}")
